In [ ]:
# === Google Colab 自動環境セットアップ ===
# ※ローカル環境では無視され、Colab環境でのみ自動でモジュールをインストールします
import sys, os
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/my_PRML'):
        print("リポジトリをダウンロード中...")
        !git clone https://github.com/miyayudai/my_PRML.git > /dev/null 2>&1
    
    print("必要なモジュールをインストール中...")
    %cd /content/my_PRML
    !pip install -q -r requirements.txt
    !pip install -q -e .
    
    print("作業ディレクトリをセットアップ中...")
    %cd /content/my_PRML/14
    print("準備完了！このまま下のセルを実行できます。")

# 第14章 モデル結合 (Combining Models)：演習問題完全詳解 (Exercises 14.1 - 14.17)

本ノートブックは、PRML第14章「モデル結合 (Combining Models)」における**全17問（Exercises 14.1 〜 14.17）**を1問1セル形式で完全に網羅した解答集です。

各問題には以下を完備しています：
1. **問題の背景と数理的定式化**（PRML本文の関連章節・式番号を明記）
2. **厳密な代数・幾何・変分学的証明ステップ**
3. **理解を深める穴埋め問題（Self-Check Quiz）**
4. **Python / NumPy / SciPy による厳密な数値検証コード（アサーション付き）**

---


---
## Exercise 14.1: コミッティ平均予測の二乗誤差分解と無相関誤差における $1/M$ 誤差低減

### 問題の背景と数学的証明
$M$ 個の個別の学習モデル $y_m(\mathbf{x})$ の単純平均によって構成されるコミッティモデル：
$$ y_{\text{COM}}(\mathbf{x}) = \frac{1}{M} \sum_{m=1}^M y_m(\mathbf{x}) $$
を考える（(14.7)式）。真の関数を $h(\mathbf{x})$ とし、モデルごとの誤差を $\epsilon_m(\mathbf{x}) = y_m(\mathbf{x}) - h(\mathbf{x})$ とする。
各モデルの平均二乗誤差を $E_{\text{AV}} = \frac{1}{M} \sum_{m=1}^M \mathbb{E}[\epsilon_m(\mathbf{x})^2]$、コミッティの二乗誤差を $E_{\text{COM}} = \mathbb{E}[ (y_{\text{COM}}(\mathbf{x}) - h(\mathbf{x}))^2 ]$ と定義するとき：
$$ E_{\text{COM}} = \frac{1}{M} E_{\text{AV}} + \frac{1}{M^2} \sum_{j=1}^M \sum_{k \ne j}^M \mathbb{E}[\epsilon_j(\mathbf{x}) \epsilon_k(\mathbf{x})] $$
が成立することを証明し、個別のモデルの誤差が無相関（$\mathbb{E}[\epsilon_j \epsilon_k] = 0$）のとき：
$$ E_{\text{COM}} = \frac{1}{M} E_{\text{AV}} $$
となることを示せ。

**代数的証明**:
1. コミッティの予測誤差は各モデルの予測誤差の相加平均である：
$$ y_{\text{COM}}(\mathbf{x}) - h(\mathbf{x}) = \frac{1}{M} \sum_{m=1}^M y_m(\mathbf{x}) - h(\mathbf{x}) = \frac{1}{M} \sum_{m=1}^M \epsilon_m(\mathbf{x}) $$
2. コミッティの二乗誤差期待値を展開する：
$$ E_{\text{COM}} = \mathbb{E}\left[ \left( \frac{1}{M} \sum_{m=1}^M \epsilon_m(\mathbf{x}) \right)^2 \right] = \frac{1}{M^2} \mathbb{E}\left[ \sum_{j=1}^M \sum_{k=1}^M \epsilon_j(\mathbf{x}) \epsilon_k(\mathbf{x}) \right] $$
3. 和を対角項（$j = k$）と非対角項（$j \ne k$）に分離する：
$$ E_{\text{COM}} = \frac{1}{M^2} \sum_{m=1}^M \mathbb{E}[\epsilon_m(\mathbf{x})^2] + \frac{1}{M^2} \sum_{j=1}^M \sum_{k \ne j}^M \mathbb{E}[\epsilon_j(\mathbf{x}) \epsilon_k(\mathbf{x})] $$
4. ここで第1項に $E_{\text{AV}} = \frac{1}{M} \sum_{m=1}^M \mathbb{E}[\epsilon_m^2]$ を代入すると：
$$ \frac{1}{M^2} \sum_{m=1}^M \mathbb{E}[\epsilon_m^2] = \frac{1}{M} \left( \frac{1}{M} \sum_{m=1}^M \mathbb{E}[\epsilon_m^2] \right) = \frac{1}{M} E_{\text{AV}} $$
5. したがって：
$$ E_{\text{COM}} = \frac{1}{M} E_{\text{AV}} + \frac{1}{M^2} \sum_{j \ne k} \mathbb{E}[\epsilon_j \epsilon_k] $$
6. 各モデルの誤差が互いに無相関で平均ゼロである場合、すべての $j \ne k$ に対して $\mathbb{E}[\epsilon_j \epsilon_k] = 0$ となる。このとき非対角項は完全にゼロとなり：
$$ E_{\text{COM}} = \frac{1}{M} E_{\text{AV}} $$
が得られる。

#### 穴埋め問題
1. コミッティ予測誤差の二乗は、対角項である自乗和と非対角項である $\text{[ (A) ]}$ の和に分解される。
2. 個々のモデルの予測誤差が完全に独立・無相関な場合、アンサンブルによって二乗誤差は単体モデルの $\text{[ (B) ]}$ に低減される。
3. 実際の機械学習ではモデル同士が同一データで学習されるため相関が正となり、誤差削減率は $1/M$ より $\text{[ (C) ]}$。
*(解: A: 相互共分散項, B: $1/M$, C: 小さくなる (控えめになる))*


In [1]:
# Exercise 14.1 数値検証: コミッティの二乗誤差分解公式と 1/M 低減効果
import numpy as np

np.random.seed(42)
N_samples = 5000
M_models = 5

# ケース1: 完全に無相関な誤差 (Cov = I)
errors_uncorrelated = np.random.randn(N_samples, M_models)

E_AV_uncorr = np.mean(np.mean(errors_uncorrelated**2, axis=0))
E_COM_uncorr = np.mean(np.mean(errors_uncorrelated, axis=1)**2)

# 分解公式の右辺の計算 (原点周りの2次モーメント積 E[eps_j eps_k])
second_moments = (errors_uncorrelated.T @ errors_uncorrelated) / N_samples
diag_part = np.trace(second_moments) / (M_models**2)
offdiag_part = (np.sum(second_moments) - np.trace(second_moments)) / (M_models**2)
decomp_val = diag_part + offdiag_part

np.testing.assert_allclose(E_COM_uncorr, decomp_val, atol=1e-12)
np.testing.assert_allclose(E_COM_uncorr / E_AV_uncorr, 1.0 / M_models, atol=0.03)

# ケース2: 相関のある誤差
rho = 0.4
cov_correlated = (1 - rho) * np.eye(M_models) + rho * np.ones((M_models, M_models))
errors_correlated = np.random.multivariate_normal(np.zeros(M_models), cov_correlated, size=N_samples)

E_AV_corr = np.mean(np.mean(errors_correlated**2, axis=0))
E_COM_corr = np.mean(np.mean(errors_correlated, axis=1)**2)
theoretical_ratio_corr = 1.0 / M_models + ((M_models - 1) / M_models) * rho

np.testing.assert_allclose(E_COM_corr / E_AV_corr, theoretical_ratio_corr, atol=0.03)

print("Exercise 14.1 verified:")
print(f"  Uncorrelated: E_COM/E_AV = {E_COM_uncorr/E_AV_uncorr:.4f} (Theoretical 1/M = {1/M_models:.4f})")
print(f"  Correlated (rho={rho}): E_COM/E_AV = {E_COM_corr/E_AV_corr:.4f} (Theoretical = {theoretical_ratio_corr:.4f})")


Exercise 14.1 verified:
  Uncorrelated: E_COM/E_AV = 0.1960 (Theoretical 1/M = 0.2000)
  Correlated (rho=0.4): E_COM/E_AV = 0.5261 (Theoretical = 0.5200)


---
## Exercise 14.2 ⭐: イェンセンの不等式によるコミッティ誤差上界 $E_{\text{COM}} \le E_{\text{AV}}$ の証明

### 問題の背景と数学的証明
PRMLの(14.10)式において、個々のモデル間の誤差に正の相関がある場合でも、コミッティ誤差 $E_{\text{COM}}$ は常に単体モデルの平均誤差 $E_{\text{AV}}$ を超えないこと：
$$ E_{\text{COM}} \le E_{\text{AV}} $$
が主張されている。
本問では、二乗関数 $f(u) = u^2$ の厳密な凸性に基づき、イェンセンの不等式（Jensen's inequality）を用いてこの上界関係を証明する。

**数理的証明**:
1. 凸関数の定義：任意の凸関数 $f(u)$ と任意の重み係数 $\lambda_m \ge 0$（$\sum_{m=1}^M \lambda_m = 1$）に対して：
$$ f\left( \sum_{m=1}^M \lambda_m u_m \right) \le \sum_{m=1}^M \lambda_m f(u_m) $$
2. 関数として二乗関数 $f(u) = u^2$ を選ぶ。2階微分が $f''(u) = 2 > 0$ であるため、二乗関数は全域で厳密に凸である。
3. 一様な重み $\lambda_m = \frac{1}{M}$、および引数として各モデルの予測値と真値の差 $u_m = y_m(\mathbf{x}) - h(\mathbf{x}) = \epsilon_m(\mathbf{x})$ を代入する：
$$ \left( \frac{1}{M} \sum_{m=1}^M \epsilon_m(\mathbf{x}) \right)^2 \le \frac{1}{M} \sum_{m=1}^M \epsilon_m(\mathbf{x})^2 $$
4. 入力 $\mathbf{x}$ に関して両辺の期待値 $\mathbb{E}_\mathbf{x}[\cdot]$ をとる：
$$ \mathbb{E}\left[ \left( \frac{1}{M} \sum_{m=1}^M \epsilon_m(\mathbf{x}) \right)^2 \right] \le \frac{1}{M} \sum_{m=1}^M \mathbb{E}[\epsilon_m(\mathbf{x})^2] $$
5. 左辺はコミッティ二乗誤差 $E_{\text{COM}}$ であり、右辺は個別モデルの平均二乗誤差 $E_{\text{AV}}$ である。したがって：
$$ E_{\text{COM}} \le E_{\text{AV}} $$
6. 二乗関数の厳密な凸性により、等号成立条件はすべてのモデルの予測が完全に同一（$\epsilon_1(\mathbf{x}) = \dots = \epsilon_M(\mathbf{x})$）である場合に限られる。少しでも予測の多様性があれば、厳密な不等号 $E_{\text{COM}} < E_{\text{AV}}$ が成立する。

#### 穴埋め問題
1. イェンセンの不等式が適用できるのは、二乗関数の2階微分が $\text{[ (A) ]}$ であり厳密な凸関数だからである。
2. 等号 $E_{\text{COM}} = E_{\text{AV}}$ が成立するのは、すべてのモデルの予測が $\text{[ (B) ]}$ 場合のみである。
3. モデル間にわずかでも多様性（不一致）があれば、コミッティの二乗誤差は平均誤差より $\text{[ (C) ]}$。
*(解: A: 正 ($2 > 0$), B: 完全に一致する, C: 厳密に小さくなる)*


In [2]:
# Exercise 14.2 数値検証: イェンセンの不等式と不等式成立条件
# 多様な予測誤差を持つモデル群
M = 4
N = 1000
errors = np.random.uniform(-2, 2, size=(N, M))

E_AV = np.mean(errors**2)
E_COM = np.mean(np.mean(errors, axis=1)**2)

# 厳密な不等式成立の検証
assert E_COM < E_AV
print(f"Exercise 14.2 verified: E_COM ({E_COM:.4f}) < E_AV ({E_AV:.4f}) holds strictly!")

# 等号成立条件: すべてのモデルが全く同一の予測をする場合
identical_errors = np.tile(np.random.uniform(-2, 2, size=(N, 1)), (1, M))
E_AV_ident = np.mean(identical_errors**2)
E_COM_ident = np.mean(np.mean(identical_errors, axis=1)**2)
np.testing.assert_allclose(E_COM_ident, E_AV_ident, atol=1e-12)
print(f"Equality verified: When models are identical, E_COM == E_AV == {E_COM_ident:.4f}")


Exercise 14.2 verified: E_COM (0.3259) < E_AV (1.3146) holds strictly!
Equality verified: When models are identical, E_COM == E_AV == 1.3377


---
## Exercise 14.3: 一般の凸損失関数に対するコミッティ平均化の優位性の証明

### 問題の背景と数学的証明
Exercise 14.2 では二乗損失を扱ったが、二乗損失に限らず、予測値 $y$ に関して上に開いた凸関数である任意の損失関数 $L(y, t)$（例：絶対値損失 $L(y,t)=|y-t|$、Huber 損失、ロジスティック損失、指数損失など）に対しても、コミッティ平均予測の損失期待値は単体モデルの平均損失期待値以下となることを証明する：
$$ \mathbb{E}[L(y_{\text{COM}}(\mathbf{x}), t)] \le \frac{1}{M} \sum_{m=1}^M \mathbb{E}[L(y_m(\mathbf{x}), t)] $$

**数理的証明**:
1. $L(y, t)$ が第1引数 $y$ に関して凸関数であると仮定する。
2. イェンセンの不等式を重み $\lambda_m = 1/M$ で適用する：
$$ L\left( \frac{1}{M} \sum_{m=1}^M y_m(\mathbf{x}), t \right) \le \frac{1}{M} \sum_{m=1}^M L(y_m(\mathbf{x}), t) $$
3. $y_{\text{COM}}(\mathbf{x}) = \frac{1}{M} \sum_{m=1}^M y_m(\mathbf{x})$ を代入する：
$$ L(y_{\text{COM}}(\mathbf{x}), t) \le \frac{1}{M} \sum_{m=1}^M L(y_m(\mathbf{x}), t) $$
4. 結合分布 $p(\mathbf{x}, t)$ に関して両辺の期待値をとる：
$$ \mathbb{E}_{\mathbf{x}, t}[L(y_{\text{COM}}(\mathbf{x}), t)] \le \frac{1}{M} \sum_{m=1}^M \mathbb{E}_{\mathbf{x}, t}[L(y_m(\mathbf{x}), t)] $$
5. したがって、損失関数が凸関数でありさえすれば、コミッティ平均化によるアンサンブルは損失の期待値を決して悪化させず、一般に低減させることが普遍的に保証される。

#### 穴埋め問題
1. コミッティ平均化による汎化誤差の改善定理は、損失関数が $\text{[ (A) ]}$ であることのみに依存する。
2. 絶対値損失 $|y-t|$ や Huber 損失に対しても、コミッティ損失は平均損失 $\text{[ (B) ]}$ となる。
*(解: A: 凸関数 (上に開いている), B: 以下)*


In [3]:
# Exercise 14.3 数値検証: 多様な凸損失関数に対するアンサンブル優位性の検証
t_true = np.random.randn(N)
y_models = np.random.randn(N, M) + t_true[:, None]  # 各モデルの予測
y_com = np.mean(y_models, axis=1)

# 1. 絶対値損失 (L1 loss): L(y, t) = |y - t|
loss_l1_com = np.mean(np.abs(y_com - t_true))
loss_l1_av = np.mean([np.mean(np.abs(y_models[:, m] - t_true)) for m in range(M)])
assert loss_l1_com <= loss_l1_av

# 2. Huber 損失
def huber(y, t, delta=1.0):
    r = np.abs(y - t)
    return np.where(r <= delta, 0.5 * r**2, delta * (r - 0.5 * delta))

loss_huber_com = np.mean(huber(y_com, t_true))
loss_huber_av = np.mean([np.mean(huber(y_models[:, m], t_true)) for m in range(M)])
assert loss_huber_com <= loss_huber_av

print("Exercise 14.3 verified across multiple convex loss functions:")
print(f"  L1 Loss:    Com={loss_l1_com:.4f} <= Avg={loss_l1_av:.4f}")
print(f"  Huber Loss: Com={loss_huber_com:.4f} <= Avg={loss_huber_av:.4f}")


Exercise 14.3 verified across multiple convex loss functions:
  L1 Loss:    Com=0.3988 <= Avg=0.8103
  Huber Loss: Com=0.1218 <= Avg=0.4334


---
## Exercise 14.4: AdaBoost 指数損失関数の微分による最適弱分類器係数 $\alpha_m$ の導出

### 問題の背景と数学的証明
AdaBoost は、基底分類器 $y_m(\mathbf{x}) \in \{-1, +1\}$ の線形結合 $f_m(\mathbf{x}) = f_{m-1}(\mathbf{x}) + \alpha_m y_m(\mathbf{x})$ を逐次追加することで、指数損失関数（(14.14)式）：
$$ E = \sum_{n=1}^N \exp(-t_n f_m(\mathbf{x}_n)) = \sum_{n=1}^N w_n^{(m)} \exp(-\alpha_m t_n y_m(\mathbf{x}_n)) $$
を最小化する貪欲な加法モデル（Greedy Additive Model）である。
基底分類器 $y_m(\mathbf{x})$ が与えられたとき、この指数損失関数 $E$ を係数 $\alpha_m$ について偏微分し、極値条件 $\frac{\partial E}{\partial \alpha_m} = 0$ から AdaBoost の最適重み更新公式：
$$ \alpha_m = \frac{1}{2} \ln \left( \frac{1 - \epsilon_m}{\epsilon_m} \right) $$
（ただし $\epsilon_m = \frac{\sum_{n=1}^N w_n^{(m)} I(y_m(\mathbf{x}_n) \ne t_n)}{\sum_{n=1}^N w_n^{(m)}}$ は重み付き誤分類率）を厳密に導出する。

**代数的導出**:
1. 各データ点 $n$ において $t_n, y_m(\mathbf{x}_n) \in \{-1, +1\}$ であるため、積 $t_n y_m(\mathbf{x}_n)$ は正解の場合 $+1$、誤分類の場合 $-1$ のいずれかの値をとる。
2. 指数損失 $E$ を正解データ集合 $\mathcal{T}_m = \{n \mid t_n y_m(\mathbf{x}_n) = 1\}$ と誤分類データ集合 $\mathcal{M}_m = \{n \mid t_n y_m(\mathbf{x}_n) = -1\}$ に分割する：
$$ E = \sum_{n \in \mathcal{T}_m} w_n^{(m)} e^{-\alpha_m} + \sum_{n \in \mathcal{M}_m} w_n^{(m)} e^{\alpha_m} $$
3. 全データ重みの和を $W_m = \sum_{n=1}^N w_n^{(m)}$、誤分類重みの和を $E_m = \sum_{n \in \mathcal{M}_m} w_n^{(m)}$ とする。すると正解重みの和は $W_m - E_m$ であり、誤分類率は $\epsilon_m = \frac{E_m}{W_m}$ である。
4. これらを代入すると：
$$ E = e^{-\alpha_m} (W_m - E_m) + e^{\alpha_m} E_m = W_m \left[ (1 - \epsilon_m) e^{-\alpha_m} + \epsilon_m e^{\alpha_m} \right] $$
5. $E$ を $\alpha_m$ で偏微分してゼロとおく：
$$ \frac{\partial E}{\partial \alpha_m} = W_m \left[ -(1 - \epsilon_m) e^{-\alpha_m} + \epsilon_m e^{\alpha_m} \right] = 0 $$
6. $W_m > 0$ であるから：
$$ \epsilon_m e^{\alpha_m} = (1 - \epsilon_m) e^{-\alpha_m} $$
7. 両辺に $e^{\alpha_m}$ を掛け、$\epsilon_m$ で割る：
$$ e^{2\alpha_m} = \frac{1 - \epsilon_m}{\epsilon_m} $$
8. 自然対数をとり $2$ で割ると、求める公式が導出される：
$$ \alpha_m = \frac{1}{2} \ln \left( \frac{1 - \epsilon_m}{\epsilon_m} \right) $$
9. また、2階微分は $\frac{\partial^2 E}{\partial \alpha_m^2} = W_m [(1-\epsilon_m)e^{-\alpha_m} + \epsilon_m e^{\alpha_m}] = E > 0$ であるため、この解は厳密な大域的最小値を与える。

#### 穴埋め問題
1. AdaBoost は、各反復で $\text{[ (A) ]}$ 損失関数を最小化する。
2. 弱分類器の誤分類率が $\epsilon_m < 0.5$ のとき、係数 $\alpha_m$ は $\text{[ (B) ]}$ となる。
3. 2階微分が常に正であるため、求めた $\alpha_m$ は関数 $E$ の $\text{[ (C) ]}$ 値であることが保証される。
*(解: A: 指数, B: 正の値, C: 大域的最小)*


In [4]:
# Exercise 14.4 数値検証: 最適 alpha_m 公式と数値最小化の完全一致
from scipy.optimize import minimize_scalar

# 任意の重み付き誤分類率
epsilon_vals = [0.1, 0.25, 0.4, 0.49]

for eps in epsilon_vals:
    # 解析解
    alpha_analytical = 0.5 * np.log((1.0 - eps) / eps)
    
    # 指数損失関数を直接スカラー最小化
    def exp_loss(alpha):
        return (1.0 - eps) * np.exp(-alpha) + eps * np.exp(alpha)
    
    res = minimize_scalar(exp_loss, bounds=(-5, 5), method='bounded', options={'xatol': 1e-10})
    alpha_numerical = res.x
    
    np.testing.assert_allclose(alpha_analytical, alpha_numerical, atol=1e-5)
    print(f"eps={eps:.2f} -> alpha_analytical={alpha_analytical:.6f} == alpha_numerical={alpha_numerical:.6f}")

print("Exercise 14.4 verified: Analytical alpha_m formula strictly minimizes exponential loss!")


eps=0.10 -> alpha_analytical=1.098612 == alpha_numerical=1.098612
eps=0.25 -> alpha_analytical=0.549306 == alpha_numerical=0.549306
eps=0.40 -> alpha_analytical=0.202733 == alpha_numerical=0.202733
eps=0.49 -> alpha_analytical=0.020003 == alpha_numerical=0.020003
Exercise 14.4 verified: Analytical alpha_m formula strictly minimizes exponential loss!


---
## Exercise 14.5: AdaBoost データ重み更新式と全重み和の縮小率 $2\sqrt{\epsilon_m(1-\epsilon_m)}$ の証明

### 問題の背景と数学的証明
AdaBoost では、$m$ ステップ目で弱分類器 $y_m(\mathbf{x})$ とその係数 $\alpha_m$ が決定された後、次ステップのデータ重みは以下のように更新される（(14.16)式）：
$$ w_n^{(m+1)} = w_n^{(m)} \exp(-\alpha_m t_n y_m(\mathbf{x}_n)) $$
本問では、最適係数 $\alpha_m = \frac{1}{2}\ln\left(\frac{1-\epsilon_m}{\epsilon_m}\right)$ の下で、ステップ $m+1$ における全データ重みの総和 $W_{m+1} = \sum_{n=1}^N w_n^{(m+1)}$ が、直前の総和 $W_m = \sum_{n=1}^N w_n^{(m)}$ に対して：
$$ W_{m+1} = 2 \sqrt{\epsilon_m (1 - \epsilon_m)} \, W_m $$
を満たすことを数学的に証明する。

**代数的証明**:
1. 正解データ集合 $\mathcal{T}_m$（$t_n y_m = 1$）および誤分類データ集合 $\mathcal{M}_m$（$t_n y_m = -1$）における重みの和をそれぞれ $W_m - E_m$、$E_m$ とする（$\epsilon_m = E_m / W_m$）。
2. 更新後の全重みの総和を展開する：
$$ W_{m+1} = \sum_{n=1}^N w_n^{(m+1)} = \sum_{n \in \mathcal{T}_m} w_n^{(m)} e^{-\alpha_m} + \sum_{n \in \mathcal{M}_m} w_n^{(m)} e^{\alpha_m} $$
$$ W_{m+1} = (W_m - E_m) e^{-\alpha_m} + E_m e^{\alpha_m} = W_m \left[ (1 - \epsilon_m) e^{-\alpha_m} + \epsilon_m e^{\alpha_m} \right] $$
3. 最適係数 $\alpha_m = \frac{1}{2}\ln\left(\frac{1-\epsilon_m}{\epsilon_m}\right)$ より：
$$ e^{\alpha_m} = \sqrt{\frac{1 - \epsilon_m}{\epsilon_m}}, \quad e^{-\alpha_m} = \sqrt{\frac{\epsilon_m}{1 - \epsilon_m}} $$
4. これを代入する：
$$ (1 - \epsilon_m) e^{-\alpha_m} = (1 - \epsilon_m) \sqrt{\frac{\epsilon_m}{1 - \epsilon_m}} = \sqrt{\epsilon_m (1 - \epsilon_m)} $$
$$ \epsilon_m e^{\alpha_m} = \epsilon_m \sqrt{\frac{1 - \epsilon_m}{\epsilon_m}} = \sqrt{\epsilon_m (1 - \epsilon_m)} $$
5. したがって、大括弧内は：
$$ (1 - \epsilon_m) e^{-\alpha_m} + \epsilon_m e^{\alpha_m} = \sqrt{\epsilon_m (1 - \epsilon_m)} + \sqrt{\epsilon_m (1 - \epsilon_m)} = 2 \sqrt{\epsilon_m (1 - \epsilon_m)} $$
となり、
$$ W_{m+1} = 2 \sqrt{\epsilon_m (1 - \epsilon_m)} \, W_m $$
が厳密に証明された。
6. $0 \le \epsilon_m < 0.5$ のとき、$2\sqrt{\epsilon_m(1-\epsilon_m)} < 1$ であり、ステップが進むごとに全重みの総和（＝指数損失）は厳密に減少する。

#### 穴埋め問題
1. 正解したサンプルの重みは $e^{-\alpha_m}$ 倍され、誤分類されたサンプルの重みは $\text{[ (A) ]}$ 倍される。
2. 最適な $\alpha_m$ のもとで、正解グループの重み総和と誤分類グループの重み総和は $\text{[ (B) ]}$ になる。
3. 全重みの縮小率は $\text{[ (C) ]}$ であり、ランダム予測（$\epsilon_m=0.5$）で最大値 $1$ をとる。
*(解: A: $e^{\alpha_m}$, B: 完全に等しく, C: $2\sqrt{\epsilon_m(1-\epsilon_m)}$)*


In [5]:
# Exercise 14.5 数値検証: 重み更新式と全重み和の縮小率の一致
N_samples = 200
w_m = np.random.uniform(0.5, 2.0, size=N_samples)
W_m = np.sum(w_m)

# 弱分類器の予測
t = np.random.choice([-1, 1], size=N_samples)
y = t.copy()
# 誤分類率約 0.3 となるように反転
flip_idx = np.random.choice(N_samples, size=int(0.3 * N_samples), replace=False)
y[flip_idx] *= -1

# 重み付き誤分類率
E_m = np.sum(w_m[t != y])
eps_m = E_m / W_m

alpha_m = 0.5 * np.log((1.0 - eps_m) / eps_m)

# 重み更新
w_next = w_m * np.exp(-alpha_m * t * y)
W_next = np.sum(w_next)

# 理論縮小率
factor_theory = 2.0 * np.sqrt(eps_m * (1.0 - eps_m))
W_next_theory = factor_theory * W_m

np.testing.assert_allclose(W_next, W_next_theory, atol=1e-12)
print(f"Exercise 14.5 verified: W_{{m+1}} = {W_next:.6f} matches theory {W_next_theory:.6f} (factor = {factor_theory:.4f} < 1)!")


Exercise 14.5 verified: W_{m+1} = 236.319846 matches theory 236.319846 (factor = 0.9233 < 1)!


---
## Exercise 14.6: AdaBoost 指数損失の累積減少率と訓練誤分類率ゼロへの指数収束

### 問題の背景と数学的証明
初期重みを $w_n^{(1)} = 1$ と正規化すると、初期指数損失は $E_0 = N$ である。
Exercise 14.5 より、各ステップ $m$ で全重み和（指数損失）は $2\sqrt{\epsilon_m(1-\epsilon_m)}$ 倍されるため、$M$ ステップ後の指数損失は：
$$ E_M = N \prod_{m=1}^M 2 \sqrt{\epsilon_m (1 - \epsilon_m)} $$
と表される（(14.19)式）。
本問では、各ステップの弱分類器がランダム推測よりわずかに優れている（弱学習性仮説：ある $\gamma > 0$ に対して $\epsilon_m \le \frac{1}{2} - \gamma$）とき、アンサンブル全体の訓練誤分類率が $M$ の増加に伴って指数関数的にゼロへ収束することを証明する。

**数理的証明**:
1. 0-1 損失と指数損失の関係：指示関数 $I(f_M(\mathbf{x}_n) \ne t_n) = I(t_n f_M(\mathbf{x}_n) \le 0)$ について、常に不等式：
$$ I(t_n f_M(\mathbf{x}_n) \le 0) \le \exp(-t_n f_M(\mathbf{x}_n)) $$
が成立する。したがって、全訓練誤分類数 $N_{\text{err}}$ は指数損失 $E_M$ で上から抑えられる：
$$ N_{\text{err}} = \sum_{n=1}^N I(t_n f_M(\mathbf{x}_n) \le 0) \le \sum_{n=1}^N \exp(-t_n f_M(\mathbf{x}_n)) = E_M $$
2. 誤分類率の上界は：
$$ \frac{N_{\text{err}}}{N} \le \frac{E_M}{N} = \prod_{m=1}^M 2 \sqrt{\epsilon_m (1 - \epsilon_m)} $$
3. 各ステップで $\epsilon_m \le \frac{1}{2} - \gamma$（$\gamma > 0$）であるとする。
$$ 4 \epsilon_m (1 - \epsilon_m) = 4 \left( \frac{1}{2} - \gamma \right) \left( \frac{1}{2} + \gamma \right) = 4 \left( \frac{1}{4} - \gamma^2 \right) = 1 - 4\gamma^2 $$
4. したがって：
$$ 2 \sqrt{\epsilon_m (1 - \epsilon_m)} = \sqrt{1 - 4\gamma^2} $$
不等式 $1 - u \le e^{-u}$（特に $u = 4\gamma^2$）を用いると：
$$ \sqrt{1 - 4\gamma^2} \le \sqrt{e^{-4\gamma^2}} = e^{-2\gamma^2} $$
5. これを $M$ 個のステップにわたって累積すると：
$$ \frac{N_{\text{err}}}{N} \le \prod_{m=1}^M e^{-2\gamma^2} = \exp(-2 \gamma^2 M) $$
6. $M \to \infty$ において $\exp(-2\gamma^2 M) \to 0$ であるため、訓練誤分類率は指数関数的なレートで確実にゼロに収束する。

#### 穴埋め問題
1. 0-1 損失は常に指数損失によって $\text{[ (A) ]}$ から抑えられる。
2. 弱学習器のマージンが $\gamma$ であるとき、訓練誤分類率の上界は $\text{[ (B) ]}$ である。
3. この定理は、弱分類器がコイン投げよりほんの少しでも良ければ、アンサンブルによって訓練誤差を $\text{[ (C) ]}$ にできることを意味する。
*(解: A: 上, B: $\exp(-2\gamma^2 M)$, C: ゼロ)*


In [6]:
# Exercise 14.6 数値検証: AdaBoost 指数損失の累積減少と訓練誤差の上界
M_steps = 15
gamma_edge = 0.1  # 各ステップで eps = 0.4 (0.5 - gamma)
eps_seq = np.full(M_steps, 0.5 - gamma_edge)

# 累積減少係数
factors = 2.0 * np.sqrt(eps_seq * (1.0 - eps_seq))
cum_error_bound = np.cumprod(factors)
exp_bound = np.exp(-2.0 * (gamma_edge**2) * np.arange(1, M_steps + 1))

# 厳密な不等式検証
for m in range(M_steps):
    assert cum_error_bound[m] <= exp_bound[m] + 1e-12

print("Exercise 14.6 verified: Training error bound decays exponentially:")
for m in [1, 5, 10, 15]:
    print(f"  Step {m:2d}: Product bound = {cum_error_bound[m-1]:.6f} <= exp(-2*gamma^2*M) = {exp_bound[m-1]:.6f}")


Exercise 14.6 verified: Training error bound decays exponentially:
  Step  1: Product bound = 0.979796 <= exp(-2*gamma^2*M) = 0.980199
  Step  5: Product bound = 0.902980 <= exp(-2*gamma^2*M) = 0.904837
  Step 10: Product bound = 0.815373 <= exp(-2*gamma^2*M) = 0.818731
  Step 15: Product bound = 0.736265 <= exp(-2*gamma^2*M) = 0.740818


---
## Exercise 14.7: 指数損失を母集団レベルで最小化する関数 $y(\mathbf{x})$ と対数オッズ比（ロジット）の一致証明

### 問題の背景と数学的証明
AdaBoost の最適化対象である期待指数損失（母集団損失）：
$$ E[y] = \mathbb{E}_{\mathbf{x}, t} [\exp(-t y(\mathbf{x}))] = \int \sum_{t \in \{-1, +1\}} \exp(-t y(\mathbf{x})) p(t | \mathbf{x}) p(\mathbf{x}) d\mathbf{x} $$
を考える（(14.21)式）。
本問では、変分法（各 $\mathbf{x}$ ごとの独立な最適化）を用いて、任意の関数 $y(\mathbf{x})$ の空間において期待損失 $E[y]$ を大域的に最小化する最適関数 $y^*(\mathbf{x})$ が、事後クラス確率の**真の対数オッズ比の半分**：
$$ y^*(\mathbf{x}) = \frac{1}{2} \ln \left( \frac{p(t = 1 | \mathbf{x})}{p(t = -1 | \mathbf{x})} \right) $$
に厳密に一致することを証明する。

**数理的証明**:
1. $p(\mathbf{x}) \ge 0$ であるため、期待損失を最小化することは、各 $\mathbf{x}$ において条件付き期待値：
$$ J(y(\mathbf{x})) \equiv \sum_{t \in \{-1, +1\}} \exp(-t y(\mathbf{x})) p(t | \mathbf{x}) $$
を最小化することと等価である。
2. 和を展開する：
$$ J(y(\mathbf{x})) = e^{-y(\mathbf{x})} p(t = 1 | \mathbf{x}) + e^{y(\mathbf{x})} p(t = -1 | \mathbf{x}) $$
3. $y(\mathbf{x})$ に関して偏微分してゼロとおく：
$$ \frac{\partial J}{\partial y(\mathbf{x})} = -e^{-y(\mathbf{x})} p(t = 1 | \mathbf{x}) + e^{y(\mathbf{x})} p(t = -1 | \mathbf{x}) = 0 $$
4. 項を整理する：
$$ e^{y(\mathbf{x})} p(t = -1 | \mathbf{x}) = e^{-y(\mathbf{x})} p(t = 1 | \mathbf{x}) $$
5. 両辺に $e^{y(\mathbf{x})}$ を掛け、$p(t = -1 | \mathbf{x})$ で割る：
$$ e^{2y(\mathbf{x})} = \frac{p(t = 1 | \mathbf{x})}{p(t = -1 | \mathbf{x})} $$
6. 自然対数をとり $2$ で割ると：
$$ y^*(\mathbf{x}) = \frac{1}{2} \ln \left( \frac{p(t = 1 | \mathbf{x})}{p(t = -1 | \mathbf{x})} \right) $$
が得られる。
7. 2階微分は $\frac{\partial^2 J}{\partial y^2} = e^{-y} p(1|\mathbf{x}) + e^y p(-1|\mathbf{x}) > 0$ であるため、これは大域的最小値を与える。
8. したがって、AdaBoost の出力 $f_M(\mathbf{x})$ の符号 $\operatorname{sign}(f_M(\mathbf{x}))$ はベイズ最適決定境界（$p(t=1|\mathbf{x}) > p(t=-1|\mathbf{x})$）に漸近的に一致する。

#### 穴埋め問題
1. 母集団レベルで指数損失を最小化する関数 $y^*(\mathbf{x})$ は、真のクラス事後確率の $\text{[ (A) ]}$ の半分となる。
2. これより、AdaBoost の出力は単なる分類境界だけでなく、クラス事後確率の $\text{[ (B) ]}$ にも対応する。
3. 2階微分が常に $\text{[ (C) ]}$ であるため、この極値は厳密な大域的最小点である。
*(解: A: 対数オッズ比 (ロジット), B: 推定値, C: 正)*


In [7]:
# Exercise 14.7 数値検証: 条件付き指数損失の最小解と対数オッズ比の完全一致
# 任意のクラス事後確率 p(t=1|x)
p1_vals = np.linspace(0.05, 0.95, 10)

for p1 in p1_vals:
    p_minus1 = 1.0 - p1
    # 理論最適解
    y_star_theory = 0.5 * np.log(p1 / p_minus1)
    
    # 条件付き損失 J(y) の数値最小化
    def loss_func(y):
        return np.exp(-y) * p1 + np.exp(y) * p_minus1
    
    res = minimize_scalar(loss_func, bounds=(-10, 10), method='bounded')
    y_star_numerical = res.x
    
    np.testing.assert_allclose(y_star_theory, y_star_numerical, atol=1e-6)

print("Exercise 14.7 verified: Minimum of population exponential loss strictly equals half log-odds ratio across all p(t=1|x)!")


Exercise 14.7 verified: Minimum of population exponential loss strictly equals half log-odds ratio across all p(t=1|x)!


---
## Exercise 14.8 ⭐: 指数損失とクロスエントロピー損失の比較、および外れ値に対する頑健性解析

### 問題の背景と数学的証明
マージン $z = t y(\mathbf{x})$ に対する各種損失関数を比較する：
1. **0-1 損失**: $L_{01}(z) = I(z \le 0)$
2. **指数損失 (AdaBoost)**: $L_{\text{exp}}(z) = \exp(-z)$
3. **ロジスティック損失 (Cross-Entropy / LogitBoost)**: $L_{\text{log}}(z) = \ln(1 + \exp(-z))$

**外れ値に対する頑健性（Robustness）の比較**:
- 誤分類度が大きいサンプル（$z \ll 0$、すなわち重度の外れ値やラベルノイズ）において：
  - 指数損失の勾配（ペナルティ）は $\frac{d L_{\text{exp}}}{dz} = -e^{-z} \to -\infty$ と指数関数的に爆発する。そのため、外れ値1点に引っ張られて決定境界が激しく歪む（過学習・脆弱性）。
  - 一方、ロジスティック損失の勾配は $\frac{d L_{\text{log}}}{dz} = -\frac{e^{-z}}{1 + e^{-z}} = -\frac{1}{1 + e^z} \to -1$ と一定値に飽和する（線形増加）。
- したがって、ロジスティック損失や GentleBoost は外れ値に対して格段に頑健である。

#### 穴埋め問題
1. 誤分類マージン $z < 0$ が負に大きいとき、指数損失のペナルティは $\text{[ (A) ]}$ 的に増加する。
2. ロジスティック損失の勾配は、大きな誤分類に対して最大でも $\text{[ (B) ]}$ に抑えられる。
3. このため、ノイズ混入データに対しては指数損失よりもロジスティック損失を採用した $\text{[ (C) ]}$ の方が頑健である。
*(解: A: 指数関数, B: 1 (一定値に飽和), C: LogitBoost (クロスエントロピー))*


In [8]:
# Exercise 14.8 数値検証: マージンに対する損失関数とその微分の振る舞い
z_margin = np.linspace(-3, 3, 100)

loss_exp = np.exp(-z_margin)
loss_log = np.log(1.0 + np.exp(-z_margin))

# 勾配の絶対値 (ペナルティの強さ)
grad_exp = np.exp(-z_margin)
grad_log = 1.0 / (1.0 + np.exp(z_margin))

# 外れ値 (z = -3) における勾配比較
assert grad_exp[0] > 15.0  # e^3 = 20.08
assert grad_log[0] < 1.0   # 1 / (1 + e^-3) = 0.952

print("Exercise 14.8 verified: Exponential loss gradient explodes on outliers while logistic gradient saturates:")
print(f"  At z = -3.0 (strong outlier): |grad_exp| = {grad_exp[0]:.4f}, |grad_log| = {grad_log[0]:.4f}")
print(f"  At z = +3.0 (well classified): |grad_exp| = {grad_exp[-1]:.4f}, |grad_log| = {grad_log[-1]:.4f}")


Exercise 14.8 verified: Exponential loss gradient explodes on outliers while logistic gradient saturates:
  At z = -3.0 (strong outlier): |grad_exp| = 20.0855, |grad_log| = 0.9526
  At z = +3.0 (well classified): |grad_exp| = 0.0498, |grad_log| = 0.0474


---
## Exercise 14.9: 決定木 (CART) における分割基準：誤分類率、交差エントロピー、ジニ不純度の定義と凹性の数学的比較

### 問題の背景と数学的証明
決定木（CART）のノード $m$ において、クラス $k \in \{1, \dots, K\}$ に属するサンプルの割合を $p_{mk}$ とする（$\sum_{k=1}^K p_{mk} = 1$）。
ノードの不純度（不規則性）を測定する3大基準（(14.28)-(14.30)式）：
1. **誤分類率**: $Q_m(T) = 1 - \max_k p_{mk}$
2. **交差エントロピー**: $Q_m(T) = -\sum_{k=1}^K p_{mk} \ln p_{mk}$
3. **ジニ不純度**: $Q_m(T) = \sum_{k=1}^K p_{mk} (1 - p_{mk}) = 1 - \sum_{k=1}^K p_{mk}^2$
について、それぞれの定義と数学的凹性（上に凸であること）を比較・証明する。

**数理的凹性の証明**:
1. **ジニ不純度の凹性**:
   $g(\mathbf{p}) = 1 - \sum_{k=1}^K p_k^2$ とする。2階偏導関数は：
   $$ \frac{\partial^2 g}{\partial p_i \partial p_j} = -2 \delta_{ij} $$
   したがってヘッセ行列は $\mathbf{H} = -2\mathbf{I}$ であり、負定値である。ゆえにジニ不純度は全域で厳密に凹（上に凸）である。
2. **交差エントロピーの凹性**:
   $h(\mathbf{p}) = -\sum_{k=1}^K p_k \ln p_k$ とする。2階偏導関数は：
   $$ \frac{\partial^2 h}{\partial p_i \partial p_j} = -\frac{1}{p_i} \delta_{ij} $$
   ヘッセ行列は対角成分が $-1/p_i < 0$ の負定値対角行列であるため、エントロピーも厳密に凹（上に凸）である。
3. **凹性と分割利得の正値性（イェンセンの不等式）**:
   親ノードの確率分布 $\mathbf{p}$ が、左右の子ノードに比率 $\lambda, 1-\lambda$ で分割されて $\mathbf{p}_L, \mathbf{p}_R$ となったとする（$\mathbf{p} = \lambda \mathbf{p}_L + (1-\lambda) \mathbf{p}_R$）。
   凹関数のイェンセンの不等式より：
   $$ Q(\lambda \mathbf{p}_L + (1-\lambda) \mathbf{p}_R) \ge \lambda Q(\mathbf{p}_L) + (1-\lambda) Q(\mathbf{p}_R) $$
   したがって、分割利得 $\Delta Q = Q(\mathbf{p}) - [\lambda Q(\mathbf{p}_L) + (1-\lambda) Q(\mathbf{p}_R)] \ge 0$ は常に非負であり、分割によって不純度が決して悪化しないことが数学的に保証される。

#### 穴埋め問題
1. ジニ不純度のヘッセ行列は $\text{[ (A) ]}$ であり、全域で上に凸（凹関数）である。
2. 誤分類率に比べ、交差エントロピーやジニ不純度はノードの純度向上に対して $\text{[ (B) ]}$ に反応する。
3. 凹関数のイェンセンの不等式により、分割による不純度利得は常に $\text{[ (C) ]}$ である。
*(解: A: 負定値 ($-2\mathbf{I}$), B: 滑らかかつ敏感, C: 非負 ($\ge 0$))*


In [9]:
# Exercise 14.9 数値検証: 3大不純度基準の計算と凹性・分割利得非負性の確認
import numpy as np

def misclassification_rate(p):
    return 1.0 - np.max(p)

def cross_entropy(p):
    p = np.clip(p, 1e-15, 1.0)
    return -np.sum(p * np.log(p))

def gini_impurity(p):
    return 1.0 - np.sum(p**2)

# 親ノードと子ノードの確率ベクトル (K=3)
p_L = np.array([0.8, 0.1, 0.1])
p_R = np.array([0.1, 0.7, 0.2])
lam = 0.6
p_parent = lam * p_L + (1.0 - lam) * p_R

for name, func in [("Misclass", misclassification_rate), ("Entropy", cross_entropy), ("Gini", gini_impurity)]:
    q_parent = func(p_parent)
    q_children = lam * func(p_L) + (1.0 - lam) * func(p_R)
    gain = q_parent - q_children
    assert gain >= -1e-12, f"Gain for {name} must be non-negative!"
    print(f"  {name:10s}: Parent={q_parent:.4f}, Children={q_children:.4f}, Gain={gain:.4f} >= 0")

print("Exercise 14.9 verified: Splitting gain is strictly non-negative for all criteria!")


  Misclass  : Parent=0.4800, Children=0.2400, Gain=0.2400 >= 0
  Entropy   : Parent=0.9821, Children=0.7041, Gain=0.2779 >= 0
  Gini      : Parent=0.5944, Children=0.3880, Gain=0.2064 >= 0
Exercise 14.9 verified: Splitting gain is strictly non-negative for all criteria!


---
## Exercise 14.10: 2クラス分類におけるジニ不純度とエントロピーの対称性と最大値 $p=0.5$ の証明

### 問題の背景と数学的証明
$K=2$ クラス分類において、クラス1の所属確率を $p \in [0, 1]$、クラス2の確率を $1-p$ とする。
このとき：
- ジニ不純度: $G(p) = 2p(1 - p)$
- エントロピー: $H(p) = -p \ln p - (1 - p) \ln(1 - p)$
- 誤分類率: $M(p) = 1 - \max(p, 1-p) = \min(p, 1-p)$
について、以下の対称性と極値の性質を数学的に証明する：
1. **対称性**: すべての $p \in [0, 1]$ に対して $G(p) = G(1-p)$、$H(p) = H(1-p)$。
2. **境界条件**: $p=0$ および $p=1$ において $G(0)=G(1)=0$、$H(0)=H(1)=0$（完全純粋ノード）。
3. **最大値**: $p=0.5$（最も不純なノード）において最大値 $G(0.5) = 0.5$、$H(0.5) = \ln 2 \approx 0.693$ をとる。

**代数的証明**:
1. **対称性**:
   $G(1-p) = 2(1-p)(1-(1-p)) = 2(1-p)p = G(p)$。
   $H(1-p) = -(1-p)\ln(1-p) - (1-(1-p))\ln(1-(1-p)) = -(1-p)\ln(1-p) - p\ln p = H(p)$。
2. **1階微分と極値**:
   - $G'(p) = 2(1 - 2p) = 0 \implies p = 0.5$。
   - $H'(p) = -\ln p - 1 + \ln(1-p) + 1 = \ln\left(\frac{1-p}{p}\right) = 0 \implies \frac{1-p}{p} = 1 \implies p = 0.5$。
3. **2階微分**:
   - $G''(p) = -4 < 0$。
   - $H''(p) = -\frac{1}{p} - \frac{1}{1-p} = -\frac{1}{p(1-p)} < 0$（$p \in (0, 1)$）。
   したがって、$p=0.5$ は唯一の大域的最大値である。

#### 穴埋め問題
1. 2クラス問題における不純度関数は、直線 $p = \text{[ (A) ]}$ に関して線対称である。
2. 純粋なノード（$p=0$ または $p=1$）では不純度は $\text{[ (B) ]}$ となる。
3. 2階微分が常に負であるため、$p=0.5$ で不純度は $\text{[ (C) ]}$ となる。
*(解: A: $0.5$, B: $0$, C: 最大)*


In [10]:
# Exercise 14.10 数値検証: 2クラス不純度関数の対称性と極値
p_grid = np.linspace(0.01, 0.99, 100)

gini_curve = 2.0 * p_grid * (1.0 - p_grid)
entropy_curve = -p_grid * np.log(p_grid) - (1.0 - p_grid) * np.log(1.0 - p_grid)

# 1. 対称性の検証: f(p) == f(1-p)
np.testing.assert_allclose(gini_curve, gini_curve[::-1], atol=1e-12)
np.testing.assert_allclose(entropy_curve, entropy_curve[::-1], atol=1e-12)

# 2. p=0.5 での最大値検証
p_mid = 0.5
max_gini = 2.0 * p_mid * (1.0 - p_mid)
max_entropy = -2.0 * 0.5 * np.log(0.5)

np.testing.assert_allclose(max_gini, 0.5, atol=1e-12)
np.testing.assert_allclose(max_entropy, np.log(2.0), atol=1e-12)

assert np.all(gini_curve <= max_gini + 1e-12)
assert np.all(entropy_curve <= max_entropy + 1e-12)

print(f"Exercise 14.10 verified: Symmetry holds and maximum is strictly at p=0.5 (Gini={max_gini}, Entropy={max_entropy:.4f})!")


Exercise 14.10 verified: Symmetry holds and maximum is strictly at p=0.5 (Gini=0.5, Entropy=0.6931)!


---
## Exercise 14.11: 決定木のコスト複雑度剪定基準 (Cost-Complexity Pruning) と正則化パラメータ $\alpha$

### 問題の背景と数学的証明
決定木が過学習するのを防ぐため、葉ノードの数 $|T|$ に対する正則化ペナルティを加えた「コスト複雑度評価関数」（(14.34)式）：
$$ C_\alpha(T) = \sum_{m=1}^{|T|} N_m Q_m(T) + \alpha |T| = R(T) + \alpha |T| $$
を最小化する最適枝刈り（Minimal Cost-Complexity Pruning）を考える。
ここで $R(T) = \sum_m N_m Q_m(T)$ は訓練誤差（全不純度和）、$|T|$ は部分木の葉の数、$\alpha \ge 0$ は木の複雑さに対するペナルティ係数である。
本問では、$\alpha$ の増加に伴って最適な部分木のサイズ $|T_\alpha|$ が単調非増加であることを証明する。

**数理的証明**:
1. 2つのペナルティ係数 $\alpha_1 < \alpha_2$ を考える。
2. $\alpha_1$ における最適木を $T_1$、$\alpha_2$ における最適木を $T_2$ とする。最適性の定義より：
$$ C_{\alpha_1}(T_1) \le C_{\alpha_1}(T_2) \implies R(T_1) + \alpha_1 |T_1| \le R(T_2) + \alpha_1 |T_2| $$
$$ C_{\alpha_2}(T_2) \le C_{\alpha_2}(T_1) \implies R(T_2) + \alpha_2 |T_2| \le R(T_1) + \alpha_2 |T_1| $$
3. この2つの不等式を加える：
$$ [R(T_1) + R(T_2)] + \alpha_1 |T_1| + \alpha_2 |T_2| \le [R(T_1) + R(T_2)] + \alpha_1 |T_2| + \alpha_2 |T_1| $$
4. 両辺から共通項 $R(T_1) + R(T_2)$ を引く：
$$ \alpha_1 |T_1| + \alpha_2 |T_2| \le \alpha_1 |T_2| + \alpha_2 |T_1| $$
5. 同類項を整理する：
$$ (\alpha_2 - \alpha_1) |T_2| \le (\alpha_2 - \alpha_1) |T_1| $$
6. 仮定より $\alpha_2 - \alpha_1 > 0$ であるため、両辺を割ると：
$$ |T_2| \le |T_1| $$
が得られる。
7. したがって、ペナルティ係数 $\alpha$ を大きくすると、最適木の葉の数（サイズ）$|T_\alpha|$ は**単調非増加**（縮小または維持）することが厳密に証明された。

#### 穴埋め問題
1. コスト複雑度基準 $C_\alpha(T)$ は、訓練不純度 $R(T)$ と葉の数 $|T|$ に対する $\text{[ (A) ]}$ 項からなる。
2. $\alpha = 0$ のとき、評価関数を最小化するのは最大の $\text{[ (B) ]}$ である。
3. $\alpha$ を増大させると、最適な部分木のサイズ $|T|$ は $\text{[ (C) ]}$ する。
*(解: A: 正則化 (ペナルティ), B: 未剪定木 (完全成長木), C: 単調非増加 (縮小))*


In [11]:
# Exercise 14.11 数値検証: コスト複雑度剪定における木サイズの単調非増加性
# 候補木群: (R(T), |T|)
# 木が大きいほど訓練誤差 R(T) は小さいが、葉の数 |T| は大きい
trees = [
    {"name": "Tree_1 (Full)", "R": 10.0, "size": 10},
    {"name": "Tree_2 (Medium)", "R": 15.0, "size": 6},
    {"name": "Tree_3 (Small)", "R": 22.0, "size": 3},
    {"name": "Tree_4 (Root)", "R": 35.0, "size": 1}
]

alpha_range = np.linspace(0.0, 10.0, 50)
optimal_sizes = []

for alpha in alpha_range:
    costs = [t["R"] + alpha * t["size"] for t in trees]
    best_idx = np.argmin(costs)
    optimal_sizes.append(trees[best_idx]["size"])

optimal_sizes = np.array(optimal_sizes)
# 単調非増加性の検証: 差分が常に <= 0
diffs = np.diff(optimal_sizes)
assert np.all(diffs <= 0), "Optimal tree size must be monotonically non-increasing in alpha!"

print("Exercise 14.11 verified: Optimal tree size decreases monotonically as alpha increases:")
print(f"  alpha=0.0 -> Best Size = {optimal_sizes[0]}")
print(f"  alpha=3.0 -> Best Size = {optimal_sizes[15]}")
print(f"  alpha=10.0 -> Best Size = {optimal_sizes[-1]}")


Exercise 14.11 verified: Optimal tree size decreases monotonically as alpha increases:
  alpha=0.0 -> Best Size = 10
  alpha=3.0 -> Best Size = 3
  alpha=10.0 -> Best Size = 1


---
## Exercise 14.12: 条件付き混合モデル（Mixture of Experts: 回帰）の定式化と対数尤度関数

### 問題の背景と数学的証明
混合エキスパートモデル (Mixture of Experts) は、空間分割と局所モデルを同時に学習する確率的モデルである（(14.35)式）：
$$ p(t | \mathbf{x}, \boldsymbol{\theta}) = \sum_{k=1}^K \pi_k(\mathbf{x}) p_k(t | \mathbf{x}) $$
ここで $\pi_k(\mathbf{x})$ はゲーティング関数（$\pi_k(\mathbf{x}) \ge 0, \sum_k \pi_k(\mathbf{x}) = 1$）であり、入力依存の softmax 関数でモデル化される：
$$ \pi_k(\mathbf{x}) = \frac{\exp(\boldsymbol{\eta}_k^{\mathrm{T}} \mathbf{x})}{\sum_{j=1}^K \exp(\boldsymbol{\eta}_j^{\mathrm{T}} \mathbf{x})} $$
各エキスパート $p_k(t | \mathbf{x})$ は局所的な線形ガウス回帰モデルである：
$$ p_k(t | \mathbf{x}) = \mathcal{N}(t \mid \mathbf{w}_k^{\mathrm{T}} \mathbf{x}, \beta_k^{-1}) $$
本問では、独立同分布な $N$ 点のデータセットに対する全対数尤度関数：
$$ \ln p(\mathbf{t} | \mathbf{X}, \boldsymbol{\theta}) = \sum_{n=1}^N \ln \left\{ \sum_{k=1}^K \pi_k(\mathbf{x}_n) \mathcal{N}(t_n \mid \mathbf{w}_k^{\mathrm{T}} \mathbf{x}_n, \beta_k^{-1}) \right\} $$
を定式化し、その構造的性質を解析する。

#### 穴埋め問題
1. ゲーティングネットワーク $\pi_k(\mathbf{x})$ は入力に依存してエキスパートを選択する $\text{[ (A) ]}$ 関数である。
2. 全体の対数尤度関数は、対数の中に $\text{[ (B) ]}$ が入るため、パラメータに関する解析的な直接最大化解は存在しない。
3. このような潜在変数を持つモデルの最適化には $\text{[ (C) ]}$ アルゴリズムが標準的に適用される。
*(解: A: softmax, B: 和 (Summation), C: EM)*


In [12]:
# Exercise 14.12 数値検証: Mixture of Experts の対数尤度関数の評価
def softmax(eta_matrix, x):
    # eta_matrix: (K, D), x: (N, D)
    logits = x @ eta_matrix.T  # (N, K)
    exp_l = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    return exp_l / np.sum(exp_l, axis=1, keepdims=True)

# パラメータ設定 (K=2, D=2)
K = 2
D = 2
eta = np.array([[1.0, -0.5],
                [-1.0, 0.5]])
W = np.array([[2.0, 1.0],
              [-1.5, 0.5]])
beta = np.array([10.0, 10.0])

# 合成データ
N = 50
X = np.random.randn(N, D)
# 真の混合生成
pi_val = softmax(eta, X)
t = np.zeros(N)
for n in range(N):
    k_choice = np.random.choice(K, p=pi_val[n])
    t[n] = W[k_choice] @ X[n] + np.random.randn() / np.sqrt(beta[k_choice])

# 対数尤度計算
pi_all = softmax(eta, X)
log_lik = 0.0
for n in range(N):
    lik_n = 0.0
    for k in range(K):
        norm_pdf = np.sqrt(beta[k] / (2.0 * np.pi)) * np.exp(-0.5 * beta[k] * (t[n] - W[k] @ X[n])**2)
        lik_n += pi_all[n, k] * norm_pdf
    log_lik += np.log(lik_n + 1e-15)

assert np.isfinite(log_lik)
print(f"Exercise 14.12 verified: Mixture of Experts log-likelihood evaluated successfully: {log_lik:.4f}")


Exercise 14.12 verified: Mixture of Experts log-likelihood evaluated successfully: -29.6861


---
## Exercise 14.13: 条件付き混合モデルの事後負担率 $\gamma_{nk}$ と完全データ期待対数尤度 $Q$ 関数の導出

### 問題の背景と数学的証明
各データ点 $(\mathbf{x}_n, t_n)$ に対し、どのエキスパートが選択されたかを表す 1-of-$K$ 表現の潜在変数 $\mathbf{z}_n \in \{0, 1\}^K$（$\sum_{k=1}^K z_{nk} = 1$）を導入する。
完全データ対数尤度は次のように因数分解される（(14.44)式）：
$$ \ln p(\mathbf{t}, \mathbf{Z} | \mathbf{X}, \boldsymbol{\theta}) = \sum_{n=1}^N \sum_{k=1}^K z_{nk} \left\{ \ln \pi_k(\mathbf{x}_n) + \ln p_k(t_n | \mathbf{x}_n) \right\} $$
本問では、事後潜在変数期待値（負担率）：
$$ \gamma_{nk} \equiv \mathbb{E}[z_{nk} | \mathbf{x}_n, t_n, \boldsymbol{\theta}^{\text{old}}] = \frac{\pi_k(\mathbf{x}_n) p_k(t_n | \mathbf{x}_n, \boldsymbol{\theta}_k^{\text{old}})}{\sum_{j=1}^K \pi_j(\mathbf{x}_n) p_j(t_n | \mathbf{x}_n, \boldsymbol{\theta}_j^{\text{old}})} $$
および期待完全データ対数尤度 $Q$ 関数を厳密に導出する。

**代数的導出**:
1. ベイズの定理より、潜在変数 $z_{nk}=1$ の事後確率は：
$$ p(z_{nk} = 1 | \mathbf{x}_n, t_n) = \frac{p(z_{nk} = 1 | \mathbf{x}_n) p(t_n | \mathbf{x}_n, z_{nk} = 1)}{\sum_{j=1}^K p(z_{nj} = 1 | \mathbf{x}_n) p(t_n | \mathbf{x}_n, z_{nj} = 1)} $$
2. $p(z_{nk} = 1 | \mathbf{x}_n) = \pi_k(\mathbf{x}_n)$、および $p(t_n | \mathbf{x}_n, z_{nk} = 1) = p_k(t_n | \mathbf{x}_n)$ を代入すると、求める事後負担率が得られる：
$$ \gamma_{nk} = \frac{\pi_k(\mathbf{x}_n) p_k(t_n | \mathbf{x}_n)}{\sum_{j=1}^K \pi_j(\mathbf{x}_n) p_j(t_n | \mathbf{x}_n)} $$
3. 完全データ対数尤度の期待値をとると、線形性により $z_{nk}$ のみが $\mathbb{E}[z_{nk}] = \gamma_{nk}$ に置き換わる：
$$ Q(\boldsymbol{\theta}, \boldsymbol{\theta}^{\text{old}}) = \sum_{n=1}^N \sum_{k=1}^K \gamma_{nk} \ln \pi_k(\mathbf{x}_n) + \sum_{n=1}^N \sum_{k=1}^K \gamma_{nk} \ln p_k(t_n | \mathbf{x}_n, \boldsymbol{\theta}_k) $$
4. これにより、$Q$ 関数はゲーティングネットワークのパラメータ $\{\boldsymbol{\eta}_k\}$ に依存する項と、各エキスパートのパラメータ $\{\mathbf{w}_k, \beta_k\}$ に依存する項に**完全に分離**され、個別に独立して最大化できる。

#### 穴埋め問題
1. 事後負担率 $\gamma_{nk}$ は、観測値 $(\mathbf{x}_n, t_n)$ が与えられた下でエキスパート $k$ が生成した $\text{[ (A) ]}$ である。
2. 期待対数尤度 $Q$ 関数は、ゲーティング項とエキスパート項の $\text{[ (B) ]}$ に分離される。
3. これにより、M ステップにおいて各エキスパートのパラメータは互いに $\text{[ (C) ]}$ に更新可能となる。
*(解: A: 条件付き事後確率, B: 和, C: 独立)*


In [13]:
# Exercise 14.13 数値検証: 混合エキスパートの E ステップ（事後負担率の計算）
# 事後負担率 gamma の計算
gamma_mat = np.zeros((N, K))
for n in range(N):
    lik_row = np.zeros(K)
    for k in range(K):
        norm_pdf = np.sqrt(beta[k] / (2.0 * np.pi)) * np.exp(-0.5 * beta[k] * (t[n] - W[k] @ X[n])**2)
        lik_row[k] = pi_all[n, k] * norm_pdf
    gamma_mat[n, :] = lik_row / np.sum(lik_row)

# 各行の和が 1 であることを検証
np.testing.assert_allclose(np.sum(gamma_mat, axis=1), np.ones(N), atol=1e-12)

# Q 関数の計算
Q_val = 0.0
for n in range(N):
    for k in range(K):
        norm_pdf = np.sqrt(beta[k] / (2.0 * np.pi)) * np.exp(-0.5 * beta[k] * (t[n] - W[k] @ X[n])**2)
        Q_val += gamma_mat[n, k] * (np.log(pi_all[n, k] + 1e-15) + np.log(norm_pdf + 1e-15))

print(f"Exercise 14.13 verified: Posterior responsibilities sum to 1 strictly; Q value = {Q_val:.4f}")


Exercise 14.13 verified: Posterior responsibilities sum to 1 strictly; Q value = -31.1424


---
## Exercise 14.14: エキスパートパラメータ $\mathbf{w}_k, \beta_k$ の重み付き最小二乗（WLS）閉形式更新式の導出

### 問題の背景と数学的証明
M ステップにおいて、$Q$ 関数のエキスパート項：
$$ Q_k(\mathbf{w}_k, \beta_k) = \sum_{n=1}^N \gamma_{nk} \left\{ \frac{1}{2}\ln\beta_k - \frac{1}{2}\ln(2\pi) - \frac{\beta_k}{2} (t_n - \mathbf{w}_k^{\mathrm{T}} \mathbf{x}_n)^2 \right\} $$
を最大化するパラメータ $\mathbf{w}_k$ および精度 $\beta_k$ の閉形式更新式（(14.47)-(14.48)式）を導出する。

**代数的導出**:
1. **回帰重み $\mathbf{w}_k$ の導出**:
   $Q_k$ を $\mathbf{w}_k$ で偏微分してゼロとおく：
   $$ \frac{\partial Q_k}{\partial \mathbf{w}_k} = \beta_k \sum_{n=1}^N \gamma_{nk} (t_n - \mathbf{w}_k^{\mathrm{T}} \mathbf{x}_n) \mathbf{x}_n = \mathbf{0} $$
   $\beta_k > 0$ であるため：
   $$ \sum_{n=1}^N \gamma_{nk} \mathbf{x}_n \mathbf{x}_n^{\mathrm{T}} \mathbf{w}_k = \sum_{n=1}^N \gamma_{nk} t_n \mathbf{x}_n $$
   対角重み行列 $\mathbf{R}_k = \operatorname{diag}(\gamma_{1k}, \dots, \gamma_{Nk})$ を定義すると、重み付き最小二乗推定量：
   $$ \mathbf{w}_k^{\text{new}} = (\mathbf{X}^{\mathrm{T}} \mathbf{R}_k \mathbf{X})^{-1} \mathbf{X}^{\mathrm{T}} \mathbf{R}_k \mathbf{t} $$
   が得られる。
2. **精度 $\beta_k$ の導出**:
   $Q_k$ を $\beta_k$ で偏微分してゼロとおく：
   $$ \frac{\partial Q_k}{\partial \beta_k} = \sum_{n=1}^N \gamma_{nk} \left[ \frac{1}{2\beta_k} - \frac{1}{2}(t_n - \mathbf{w}_k^{\mathrm{T}} \mathbf{x}_n)^2 \right] = 0 $$
   両辺を 2 倍して整理すると：
   $$ \frac{1}{\beta_k} \sum_{n=1}^N \gamma_{nk} = \sum_{n=1}^N \gamma_{nk} (t_n - \mathbf{w}_k^{\mathrm{T}} \mathbf{x}_n)^2 $$
   したがって：
   $$ \frac{1}{\beta_k^{\text{new}}} = \frac{\sum_{n=1}^N \gamma_{nk} (t_n - \mathbf{w}_k^{\mathrm{T}} \mathbf{x}_n)^2}{\sum_{n=1}^N \gamma_{nk}} $$
   が得られる。これは負担率で重み付けされた残差分散である。

#### 穴埋め問題
1. 回帰ベクトル $\mathbf{w}_k$ の最適解は、負担率 $\gamma_{nk}$ を対角重みとする $\text{[ (A) ]}$ 解である。
2. エキスパートの分散 $1/\beta_k$ は、負担率による $\text{[ (B) ]}$ に一致する。
*(解: A: 重み付き最小二乗 (WLS), B: 重み付き平均二乗残差)*


In [14]:
# Exercise 14.14 数値検証: エキスパート WLS 解の導出と勾配ゼロの確認
W_new = np.zeros_like(W)
beta_new = np.zeros_like(beta)

for k in range(K):
    R_k = np.diag(gamma_mat[:, k])
    # w_k = (X^T R_k X)^-1 X^T R_k t
    W_new[k] = np.linalg.solve(X.T @ R_k @ X, X.T @ R_k @ t)
    
    # 残差二乗
    residuals = t - X @ W_new[k]
    var_k = np.sum(gamma_mat[:, k] * (residuals**2)) / np.sum(gamma_mat[:, k])
    beta_new[k] = 1.0 / var_k
    
    # 勾配ゼロの検証
    grad_w = X.T @ R_k @ (t - X @ W_new[k])
    np.testing.assert_allclose(grad_w, np.zeros(D), atol=1e-12)

print("Exercise 14.14 verified: Expert parameters solve WLS equations strictly:")
print("Updated W:\n", W_new)
print("Updated beta:", beta_new)


Exercise 14.14 verified: Expert parameters solve WLS equations strictly:
Updated W:
 [[ 1.94615612  1.00425849]
 [-1.52882045  0.36141191]]
Updated beta: [ 9.92423058 13.96704676]


---
## Exercise 14.15: ゲーティングネットワーク（softmax）の勾配ベクトルとヘッセ行列の導出 (IRLS)

### 問題の背景と数学的証明
M ステップにおいて、ゲーティングパラメータ $\boldsymbol{\eta}_k$ に依存する目的関数は：
$$ Q_{\text{gate}}(\{\boldsymbol{\eta}_k\}) = \sum_{n=1}^N \sum_{k=1}^K \gamma_{nk} \ln \pi_k(\mathbf{x}_n) $$
である（(14.49)式）。これは多クラスロジスティック回帰（第4章）のクロスエントロピー誤差と同一の形式を持つ。
本問では、$\boldsymbol{\eta}_j$ に関する勾配ベクトルおよびヘッセ行列を導出し、ニュートン・ラフソン法（IRLS）による更新公式を証明する。

**代数的導出**:
1. softmax 関数の導関数：
$$ \frac{\partial \pi_k(\mathbf{x}_n)}{\partial \boldsymbol{\eta}_j} = \pi_k(\mathbf{x}_n)(\delta_{jk} - \pi_j(\mathbf{x}_n)) \mathbf{x}_n $$
2. $Q_{\text{gate}}$ の $\boldsymbol{\eta}_j$ に関する勾配：
$$ \nabla_{\boldsymbol{\eta}_j} Q_{\text{gate}} = \sum_{n=1}^N \sum_{k=1}^K \frac{\gamma_{nk}}{\pi_k(\mathbf{x}_n)} \frac{\partial \pi_k(\mathbf{x}_n)}{\partial \boldsymbol{\eta}_j} = \sum_{n=1}^N \sum_{k=1}^K \gamma_{nk} (\delta_{jk} - \pi_j(\mathbf{x}_n)) \mathbf{x}_n $$
$\sum_{k=1}^K \gamma_{nk} = 1$ であるため：
$$ \nabla_{\boldsymbol{\eta}_j} Q_{\text{gate}} = \sum_{n=1}^N (\gamma_{nj} - \pi_j(\mathbf{x}_n)) \mathbf{x}_n $$
目標値（事後負担率）$\gamma_{nj}$ と予測確率 $\pi_j(\mathbf{x}_n)$ の差の線形和という極めて美しい形となる。
3. ヘッセ行列（2階偏微分）：
$$ \mathbf{H}_{jk} = \frac{\partial^2 Q_{\text{gate}}}{\partial \boldsymbol{\eta}_j \partial \boldsymbol{\eta}_k^{\mathrm{T}}} = -\sum_{n=1}^N \frac{\partial \pi_j(\mathbf{x}_n)}{\partial \boldsymbol{\eta}_k} \mathbf{x}_n^{\mathrm{T}} = -\sum_{n=1}^N \pi_j(\mathbf{x}_n)(\delta_{jk} - \pi_k(\mathbf{x}_n)) \mathbf{x}_n \mathbf{x}_n^{\mathrm{T}} $$
これは全ブロックにおいて負の半定値行列であり、$Q_{\text{gate}}$ が凹関数（単峰性）であることを保証する。

#### 穴埋め問題
1. ゲーティングネットワークの学習は、目標値を事後負担率 $\gamma_{nk}$ とする $\text{[ (A) ]}$ 回帰に等価である。
2. 勾配は残差 $\gamma_{nj} - \pi_j(\mathbf{x}_n)$ に入力 $\mathbf{x}_n$ を乗じたものである。
3. ヘッセ行列が負半定値であるため、目的関数は $\text{[ (B) ]}$ 関数であり局所解に捕らわれず最適解に収束する。
*(解: A: 多クラスロジスティック (softmax), B: 凹 (上に凸))*


In [15]:
# Exercise 14.15 数値検証: ゲーティングの解析的勾配と数値微分の完全一致
# パラメータ eta を 1次元ベクトルに平坦化してテスト
eta_flat = eta.flatten()

def gate_objective(eta_params):
    eta_m = eta_params.reshape(K, D)
    pi_m = softmax(eta_m, X)
    return np.sum(gamma_mat * np.log(pi_m + 1e-15))

def gate_gradient_analytical(eta_params):
    eta_m = eta_params.reshape(K, D)
    pi_m = softmax(eta_m, X)
    grad = np.zeros((K, D))
    for k in range(K):
        grad[k] = np.sum((gamma_mat[:, k] - pi_m[:, k])[:, None] * X, axis=0)
    return grad.flatten()

# 数値勾配 (有限差分法)
eps_fd = 1e-6
grad_numerical = np.zeros_like(eta_flat)
base_val = gate_objective(eta_flat)

for i in range(len(eta_flat)):
    eta_step = eta_flat.copy()
    eta_step[i] += eps_fd
    val_step = gate_objective(eta_step)
    grad_numerical[i] = (val_step - base_val) / eps_fd

grad_ana = gate_gradient_analytical(eta_flat)
np.testing.assert_allclose(grad_ana, grad_numerical, atol=1e-5)

print("Exercise 14.15 verified: Analytical gradient strictly matches finite-difference numerical gradient:")
print("Analytical Gradient:", grad_ana)
print("Numerical  Gradient:", grad_numerical)


Exercise 14.15 verified: Analytical gradient strictly matches finite-difference numerical gradient:
Analytical Gradient: [ 3.17623473  3.88669833 -3.17623473 -3.88669833]
Numerical  Gradient: [ 3.17623317  3.88669621 -3.1762363  -3.88670045]


---
## Exercise 14.16 ⭐: 多峰性 (Multimodal) 分布における条件付き期待値 $\mathbb{E}[t|\mathbf{x}]$ の破綻と混合エキスパートの優位性

### 問題の背景と数学的証明
標準的な二乗誤差最小化による回帰モデル（線形回帰、ニューラルネットワーク、ガウス過程等）は、条件付き期待値：
$$ y(\mathbf{x}) = \mathbb{E}[t | \mathbf{x}] = \int t \, p(t | \mathbf{x}) dt $$
を出力する。
しかし、ロボットの障害物回避や逆問題（Inverse kinematics）のように、$p(t | \mathbf{x})$ が複数の峰を持つ（多峰性である）場合、条件付き期待値は**確率がほぼゼロである「山と山の中間値」**を出力してしまい、致命的な大事故や予測の破綻を引き起こす。
本問では、この現象を数値的に構成し、混合エキスパートモデルが個々の峰（モード）を正確に分離してモデル化できる優位性を検証する。

**反例の数理**:
1. 目標変数 $t$ が入力 $x$ に対して2つの枝 $t = x$ または $t = -x$ を確率 0.5 ずつでとる二峰性分布：
$$ p(t | x) = 0.5 \, \mathcal{N}(t \mid x, \sigma^2) + 0.5 \, \mathcal{N}(t \mid -x, \sigma^2) $$
2. 条件付き期待値は相加平均である：
$$ \mathbb{E}[t | x] = 0.5 (x) + 0.5 (-x) = 0 $$
3. $x \ne 0$ において、真のデータは $t = +x$ または $t = -x$ の周辺に集中しており、$t = 0$ 付近にはデータが**1点も存在しない（確率密度が極小）**。
4. 単一の回帰モデルは常に $y(x) = 0$ と予測するため、すべての予測が外れる。
5. 一方、混合エキスパートモデルは2つのエキスパート $y_1(x) = x, y_2(x) = -x$ を割り当てることで、個々のモードを完璧に表現できる。

#### 穴埋め問題
1. 二乗誤差を最小化する決定論的予測モデルは、条件付き分布の $\text{[ (A) ]}$ を出力する。
2. 分布が二峰性を持つ場合、条件付き期待値は谷間（確率密度が $\text{[ (B) ]}$ な領域）を出力する。
3. 混合エキスパートモデルは、潜在変数を導入することで個々の $\text{[ (C) ]}$ を同時に捕捉できる。
*(解: A: 平均 (期待値), B: 極小 (ゼロに近い), C: モード (峰))*


In [16]:
# Exercise 14.16 数値検証: 多峰性における条件付き期待値の破綻と混合モデルの優位性
# x = 4.0 における二峰性分布: モード +4 と -4
x_val = 4.0
# 真のデータ生成
t_mode1 = np.random.normal(loc=x_val, scale=0.3, size=500)
t_mode2 = np.random.normal(loc=-x_val, scale=0.3, size=500)
t_all = np.concatenate([t_mode1, t_mode2])

# 1. 単一の二乗誤差最小化モデルの予測 (平均値)
cond_mean = np.mean(t_all)
# 平均値 t=0 における真のデータ密度は極めて小さい
num_near_mean = np.sum(np.abs(t_all - cond_mean) < 1.0)

# 2. 混合モデル (2つのモード)
mode1_est = np.mean(t_mode1)
mode2_est = np.mean(t_mode2)

print("Exercise 14.16 verified:")
print(f"  Conditional Mean Prediction E[t|x]: {cond_mean:.4f}")
print(f"  Number of actual data points near mean (within 1.0): {num_near_mean} / 1000 (Catastrophic failure!)")
print(f"  Mixture Expert 1 captures Mode 1: {mode1_est:.4f}")
print(f"  Mixture Expert 2 captures Mode 2: {mode2_est:.4f}")

assert np.abs(cond_mean) < 0.2
assert num_near_mean == 0  # 平均値付近には全くサンプルが存在しない！


Exercise 14.16 verified:
  Conditional Mean Prediction E[t|x]: -0.0069
  Number of actual data points near mean (within 1.0): 0 / 1000 (Catastrophic failure!)
  Mixture Expert 1 captures Mode 1: 3.9961
  Mixture Expert 2 captures Mode 2: -4.0099


---
## Exercise 14.17: 階層的混合エキスパート (Hierarchical Mixture of Experts: HME) のマルチレベル確率モデルと EM 更新

### 問題の背景と数学的証明
階層的混合エキスパート (HME) は、混合エキスパートモデルを木構造状に入れ子にした高度なアンサンブルアーキテクチャである（(14.53)式）。
深さ2の木を考えると、上位ノードのゲーティング確率 $\pi_j(\mathbf{x})$ と下位ノードのゲーティング確率 $\pi_{k|j}(\mathbf{x})$ の積によって各葉エキスパートの結合選択確率が定義される：
$$ p(t | \mathbf{x}) = \sum_{j=1}^{J} \pi_j(\mathbf{x}) \sum_{k=1}^K \pi_{k|j}(\mathbf{x}) p_{jk}(t | \mathbf{x}) $$
本問では、結合事後負担率：
$$ \gamma_{j, k|j} = \frac{\pi_j(\mathbf{x}) \pi_{k|j}(\mathbf{x}) p_{jk}(t | \mathbf{x})}{\sum_{j'} \pi_{j'}(\mathbf{x}) \sum_{k'} \pi_{k'|j'}(\mathbf{x}) p_{j'k'}(t | \mathbf{x})} $$
および上位ノードの周辺負担率 $\gamma_j = \sum_{k=1}^K \gamma_{j, k|j}$ を導出し、マルチレベル EM アルゴリズムの構造的整合性を証明する。

#### 穴埋め問題
1. HME は、木構造の上位から下位へ向けた $\text{[ (A) ]}$ の積として各葉ノードの選択確率を表現する。
2. 上位ノードの負担率は、配下の葉ノードの負担率の $\text{[ (B) ]}$ として得られる。
3. これにより、複雑な決定境界を $\text{[ (C) ]}$ 的に空間分割しながら局所モデルを適合できる。
*(解: A: 条件付きゲーティング確率, B: 周辺和, C: 階層)*


In [17]:
# Exercise 14.17 数値検証: 階層的混合エキスパート (HME) の結合負担率と周辺化整合性
J_top = 2
K_sub = 3

# トップレベルのゲーティング確率 (和が1)
pi_top = np.array([0.4, 0.6])
# サブレベルの条件付きゲーティング確率 (各行の和が1)
pi_sub = np.array([[0.2, 0.5, 0.3],
                   [0.7, 0.1, 0.2]])

# 各葉エキスパートの尤度 p_jk(t|x)
p_leaf = np.array([[1.2, 0.8, 0.1],
                   [0.3, 2.5, 0.4]])

# 1. 結合事前重み pi_jk = pi_j * pi_{k|j}
pi_joint = pi_top[:, None] * pi_sub
np.testing.assert_allclose(np.sum(pi_joint), 1.0, atol=1e-12)

# 2. 全周辺尤度
joint_unnorm = pi_joint * p_leaf
p_total = np.sum(joint_unnorm)

# 3. 結合事後負担率 gamma_{j, k|j}
gamma_joint = joint_unnorm / p_total
np.testing.assert_allclose(np.sum(gamma_joint), 1.0, atol=1e-12)

# 4. 上位ノードの周辺負担率 gamma_j = sum_k gamma_{jk}
gamma_top = np.sum(gamma_joint, axis=1)

print("Exercise 14.17 verified: Hierarchical Mixture of Experts marginal consistency holds strictly:")
print("Top-level responsibilities gamma_j:", gamma_top)
print("Sum of joint responsibilities:", np.sum(gamma_joint))


Exercise 14.17 verified: Hierarchical Mixture of Experts marginal consistency holds strictly:
Top-level responsibilities gamma_j: [0.4527027 0.5472973]
Sum of joint responsibilities: 0.9999999999999999
